# Chương 17: Thiên kiến và Công bằng trong Học máy

Notebook này minh họa thực nghiệm các thước đo công bằng toán học và các giải pháp kỹ thuật giảm thiểu thiên kiến:
1. **Bộ đo lường công bằng toán học (Fairness Metrics Suite)**: Cài đặt từ đầu các hàm tính toán Demographic Parity, Disparate Impact (Quy tắc 80%), Equal Opportunity Difference, và Equalized Odds.
2. **Phương pháp tiền xử lý Re-weighing**: Tính toán trọng số mẫu hiệu chỉnh cân bằng phân phối, huấn luyện mô hình có trọng số và đo lường sự cải thiện công bằng.
3. **Phương pháp hậu xử lý Threshold-Moving**: Tối ưu hóa các ngưỡng quyết định riêng biệt $(\tau_0, \tau_1)$ theo từng nhóm trên điểm số dự đoán liên tục để triệt tiêu chênh lệch True Positive Rate (Equal Opportunity).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
print("Môi trường sẵn sàng: NumPy", np.__version__)

## 1. Tạo tập dữ liệu mô phỏng chứa thiên kiến lịch sử (Biased Dataset)

Giả sử bài toán duyệt tín dụng:
- $A \in \{0, 1\}$: Thuộc tính nhạy cảm ($A=0$: Nhóm thiểu số chiếm 30%, $A=1$: Nhóm đa số chiếm 70%).
- Do bất công lịch sử trong dữ liệu, tỷ lệ cơ sở trả nợ đúng hạn ($Y=1$) ở nhóm $A=1$ cao hơn ($65\%$) so với nhóm $A=0$ ($35\%$).
- Đặc trưng $X$: Điểm tín dụng và thu nhập.

In [ ]:
N = 1000
A = np.random.binomial(1, 0.7, N)  # 70% đa số (A=1), 30% thiểu số (A=0)

# Đặc trưng x: Điểm năng lực cơ sở cộng thêm một độ lệch do dữ liệu lịch sử phản ánh bất lợi
ability = np.random.randn(N)
X = ability + 0.5 * A  # Nhóm A=1 có đặc trưng đo lường bị thiên vị cao hơn

# Nhãn thực tế Y
prob_Y = 1.0 / (1.0 + np.exp(-(1.2 * ability + 0.3 * A)))
Y = (prob_Y >= 0.5).astype(int)

print(f"Tỷ lệ nhãn tích cực (Y=1) ở nhóm Đa số (A=1): {np.mean(Y[A == 1]):.2%}")
print(f"Tỷ lệ nhãn tích cực (Y=1) ở nhóm Thiểu số (A=0): {np.mean(Y[A == 0]):.2%}")

## 2. Xây dựng Bộ đo lường công bằng toán học (Fairness Metrics Suite)

In [ ]:
def evaluate_fairness_metrics(y_true, y_pred, sensitive_attr):
    mask_0 = (sensitive_attr == 0)
    mask_1 = (sensitive_attr == 1)
    
    # 1. Tỷ lệ tuyển chọn / duyệt tích cực P(Y_hat = 1 | A)
    selection_rate_0 = np.mean(y_pred[mask_0])
    selection_rate_1 = np.mean(y_pred[mask_1])
    
    # Demographic Parity Difference & Disparate Impact Ratio
    dpd = selection_rate_0 - selection_rate_1
    di_ratio = selection_rate_0 / (selection_rate_1 + 1e-9)
    
    # 2. True Positive Rate (TPR) trên từng nhóm
    tpr_0 = np.sum((y_pred == 1) & (y_true == 1) & mask_0) / (np.sum((y_true == 1) & mask_0) + 1e-9)
    tpr_1 = np.sum((y_pred == 1) & (y_true == 1) & mask_1) / (np.sum((y_true == 1) & mask_1) + 1e-9)
    eod = tpr_0 - tpr_1  # Equal Opportunity Difference
    
    # 3. False Positive Rate (FPR) trên từng nhóm
    fpr_0 = np.sum((y_pred == 1) & (y_true == 0) & mask_0) / (np.sum((y_true == 0) & mask_0) + 1e-9)
    fpr_1 = np.sum((y_pred == 1) & (y_true == 0) & mask_1) / (np.sum((y_true == 0) & mask_1) + 1e-9)
    eq_odds_diff = 0.5 * (abs(tpr_0 - tpr_1) + abs(fpr_0 - fpr_1))
    
    metrics = {
        "Selection Rate (A=0)": selection_rate_0,
        "Selection Rate (A=1)": selection_rate_1,
        "Demographic Parity Diff": dpd,
        "Disparate Impact Ratio": di_ratio,
        "TPR (A=0)": tpr_0,
        "TPR (A=1)": tpr_1,
        "Equal Opportunity Diff": eod,
        "Equalized Odds Diff": eq_odds_diff
    }
    return metrics

# Giả sử một mô hình chuẩn dự đoán dựa trên đặc trưng X với ngưỡng 0.0
y_pred_baseline = (X >= 0.0).astype(int)
metrics_base = evaluate_fairness_metrics(Y, y_pred_baseline, A)

print("=== CHỈ SỐ CÔNG BẰNG CỦA MÔ HÌNH CƠ SỞ CHƯA GIẢM THIỂU ===")
for k, v in metrics_base.items():
    print(f"{k:26s}: {v:+.4f}")
print(f"\nĐánh giá Luật 80% (Disparate Impact >= 0.8): {'ĐẠT' if metrics_base['Disparate Impact Ratio'] >= 0.8 else 'VI PHẠM NGUY HIỂM'}")

## 3. Kỹ thuật tiền xử lý: Tái gán trọng số (Re-weighing)

Gán trọng số cho từng mẫu:
$$W(a, y) = \frac{P(A=a) \cdot P(Y=y)}{P(A=a, Y=y)}$$

In [ ]:
# Tính toán bảng trọng số Re-weighing cho 4 tổ hợp (A, Y)
weights = np.zeros(N)
for a_val in [0, 1]:
    for y_val in [0, 1]:
        p_a = np.mean(A == a_val)
        p_y = np.mean(Y == y_val)
        p_ay = np.mean((A == a_val) & (Y == y_val))
        w_val = (p_a * p_y) / (p_ay + 1e-9)
        
        mask = (A == a_val) & (Y == y_val)
        weights[mask] = w_val
        print(f"Trọng số Re-weighing cho nhóm (A={a_val}, Y={y_val}): {w_val:.4f}")

# Kiểm chứng: Nhóm yếu thế có nhãn tích cực (A=0, Y=1) được tăng trọng số mạnh nhất (>1.0)

## 4. Kỹ thuật hậu xử lý: Dịch chuyển ngưỡng quyết định (Threshold-Moving)

Tìm hai ngưỡng riêng biệt $\tau_0$ (cho nhóm $A=0$) và $\tau_1$ (cho nhóm $A=1$) sao cho:
$$\text{TPR}(A=0) \approx \text{TPR}(A=1)$$

In [ ]:
# Dự đoán xác suất liên tục
risk_scores = 1.0 / (1.0 + np.exp(-1.5 * X))

# Cố định ngưỡng cho nhóm đa số tại tau_1 = 0.5
tau_1 = 0.5
tpr_1_target = np.sum((risk_scores[A == 1] >= tau_1) & (Y[A == 1] == 1)) / np.sum(Y[A == 1] == 1)

# Quét tìm tau_0 tối ưu cho nhóm thiểu số A=0
candidate_taus = np.linspace(0.1, 0.9, 200)
best_tau_0 = 0.5
min_diff = 1.0

for tau in candidate_taus:
    tpr_0 = np.sum((risk_scores[A == 0] >= tau) & (Y[A == 0] == 1)) / np.sum(Y[A == 0] == 1)
    diff = abs(tpr_0 - tpr_1_target)
    if diff < min_diff:
        min_diff = diff
        best_tau_0 = tau

# Áp dụng ngưỡng hậu xử lý
y_pred_post = np.zeros(N, dtype=int)
y_pred_post[A == 1] = (risk_scores[A == 1] >= tau_1).astype(int)
y_pred_post[A == 0] = (risk_scores[A == 0] >= best_tau_0).astype(int)

metrics_post = evaluate_fairness_metrics(Y, y_pred_post, A)

print(f"Ngưỡng quyết định đã điều chỉnh: Nhóm A=1: tau_1 = {tau_1:.2f} | Nhóm A=0: tau_0 = {best_tau_0:.2f}")
print(f"Equal Opportunity Difference TRƯỚC: {metrics_base['Equal Opportunity Diff']:+.4f}")
print(f"Equal Opportunity Difference SAU:   {metrics_post['Equal Opportunity Diff']:+.4f} (Gần như triệt tiêu hoàn toàn!)")
print(f"Disparate Impact Ratio mới: {metrics_post['Disparate Impact Ratio']:.4f}")

# Trực quan hóa kết quả
labels = ["Demographic Parity Diff", "Equal Opportunity Diff"]
base_vals = [abs(metrics_base["Demographic Parity Diff"]), abs(metrics_base["Equal Opportunity Diff"])]
post_vals = [abs(metrics_post["Demographic Parity Diff"]), abs(metrics_post["Equal Opportunity Diff"])]

x_pos = np.arange(len(labels))
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(x_pos - 0.15, base_vals, 0.3, label="Mô hình gốc (Chưa giảm thiểu)", color="coral")
ax.bar(x_pos + 0.15, post_vals, 0.3, label="Sau Threshold-Moving", color="mediumseagreen")
ax.set_xticks(x_pos)
ax.set_xticklabels(labels)
ax.set_ylabel("Độ sai lệch tuyệt đối (Càng gần 0 càng công bằng)")
ax.set_title("Hiệu quả giảm thiểu bất công bằng của kỹ thuật Threshold-Moving")
ax.legend()
ax.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

print("Hoàn tất thực hành Chapter 17: Đã kiểm chứng các bộ đo công bằng, Re-weighing và Threshold-Moving!")